<a href="https://colab.research.google.com/github/Nitish-9k/cifar10_classification_CNN/blob/main/Cifar10Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torchvision as tv
import torch.optim as optim
from torchvision.datasets import CIFAR10

In [3]:
from torch.utils.data import DataLoader
from torchvision import transforms
transforms=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

In [4]:
traindata=CIFAR10("./data",download=True,train=True,transform=transforms)
testdata=CIFAR10("./data",download=True,train=False,transform=transforms)


In [5]:
trainloader=DataLoader(traindata,batch_size=64,shuffle=True)
testloader=DataLoader(testdata,batch_size=64,shuffle=False)

**Building CNN**

In [6]:
from torch.nn.modules.pooling import MaxPool2d
from torch.nn.modules.activation import ReLU
class CNN(nn.Module):
  def __init__(self):
    super(CNN,self).__init__()
    self.convLayers=nn.Sequential(
        nn.Conv2d(3,32,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),


    )

    self.fclayer=nn.Sequential(
        nn.Linear(128*4*4,1024),
        nn.ReLU(),
        nn.Linear(1024,512),
        nn.ReLU(),
        nn.Linear(512,10)
    )

  def forward(self,x):
    x=self.convLayers(x)
    x=x.view(x.size(0),-1) #flatten
    x=self.fclayer(x)

    return x

In [7]:
import torch.optim as optim
model=CNN()
criterion=nn.CrossEntropyLoss()
model_optimizer=optim.Adam(model.parameters())

In [9]:
# training
epochs=5

for epoch in range(epochs):
  epoch_trainig_loss=0

  for images,lables in trainloader:
    model_optimizer.zero_grad()
    output=model(images)
    loss=criterion(output,lables)
    epoch_trainig_loss+=loss.item()
    loss.backward()
    model_optimizer.step()

  print(f"{epoch/epochs} ==>>{epoch_trainig_loss/len(trainloader)}")

0.0 ==>>0.28708732619767297
0.2 ==>>0.20103408257141137
0.4 ==>>0.15039956336721894
0.6 ==>>0.118871792838873


KeyboardInterrupt: 

In [10]:
total=0
correct=0

model.eval()
with torch.no_grad():
  for images,lables in testloader:
    output=model(images)
    _,predicted=torch.max(output,1)
    total+=lables.size(0)
    correct+=(predicted==lables).sum().item()

print(correct)
print(total)

7514
10000


In [11]:
Accuracy=(correct/total)*100
print(Accuracy)

75.14
